# Milestone 3: Retrieval + Prompting Strategy (Usage Notebook)

This notebook implements the retrieval and prompting stage of the architecture after embedding/index creation.

Aligned with project docs:
- `docs/objectives.md`: grounded RAG comments for single-file diffs (<= 200 lines).
- `docs/problem_statement.md`: scope restricted to 5 violation categories; exclude functional/security/architecture issues.
- `docs/Milestone 3/Milestone-3.md`: retrieve top-k evidence, predict category, generate grounded suggestion.

## Pipeline In This Notebook
1. Load evaluation dataset + FAISS index + retrieval metadata.
2. Build query from file path + diff chunk near review line number.
3. Retrieve top-k evidence chunks using dense similarity.
4. Build a grounded prompt template with strict scope constraints.
5. (Optional) Send prompt to an LLM API, or preview prompt output only.

In [1]:
# Optional install (run once if needed)
# !pip install -q sentence-transformers faiss-cpu numpy requests

In [2]:
from pathlib import Path
import json
import os
import re
from collections import Counter

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw' / 'dataset_v1'
EMBED_DIR = ROOT / 'data' / 'processed' / 'embedding'
OUT_DIR = ROOT / 'data' / 'processed' / 'milestone3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_PATH = RAW_DIR / 'evaluation_dataset.json'
INDEX_PATH = EMBED_DIR / 'faiss_index_ip.bin'
META_PATH = EMBED_DIR / 'faiss_metadata.json'
OUT_EXAMPLES = OUT_DIR / 'retrieval_prompting_examples.json'

MODEL_NAME = 'BAAI/bge-large-en-v1.5'
TOP_K = 5
ALLOWED_CATEGORIES = [
    'indentation',
    'naming_convention',
    'unused_import',
    'mutable_default',
    'documentation_formatting',
]

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def normalize_diff_line(line: str) -> str:
    if not line:
        return ''
    if line[0] in '+- ':
        return line[1:]
    return line

def extract_chunk_text(entry: dict, target_line: int) -> str:
    selected = None
    for chunk in entry.get('diff_chunks', []):
        start = int(chunk.get('start_line', 0))
        end = int(chunk.get('end_line', 0))
        if start <= target_line <= end:
            selected = chunk
            break

    if selected is None and entry.get('diff_chunks'):
        selected = entry['diff_chunks'][0]

    if selected is None:
        return ''

    lines = [normalize_diff_line(x) for x in selected.get('diff_lines', [])]
    return '\n'.join(lines)

def build_instances(eval_dataset: list[dict]) -> list[dict]:
    instances = []
    for entry in eval_dataset:
        for review in entry.get('ground_truth_reviews', []):
            line_number = int(review.get('line_number', 0))
            chunk_text = extract_chunk_text(entry, line_number)
            query_text = f"{entry.get('file_path', '')}\n{chunk_text}"
            instances.append({
                'pr_id': entry.get('pr_id'),
                'repo': entry.get('repo'),
                'file_path': entry.get('file_path'),
                'line_number': line_number,
                'query_text': query_text,
                'gold_category': review.get('violation_category'),
                'gold_comment': review.get('review_comment', ''),
            })
    return instances

def retrieve_dense(query: str, model, index, metadata: list[dict], top_k: int = TOP_K):
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    scores, idxs = index.search(q_emb, top_k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0 or idx >= len(metadata):
            continue
        doc = metadata[idx]
        hits.append({
            'score': float(score),
            'chunk_id': doc.get('chunk_id'),
            'category': doc.get('category'),
            'source_type': doc.get('source_type'),
            'text': doc.get('text', ''),
        })
    return hits

def predict_category(retrieved: list[dict]) -> str:
    votes = [x.get('category') for x in retrieved if x.get('category') in ALLOWED_CATEGORIES]
    if not votes:
        return 'documentation_formatting'
    return Counter(votes).most_common(1)[0][0]

def build_prompt(instance: dict, retrieved: list[dict], predicted_category: str) -> str:
    evidence_lines = []
    for i, hit in enumerate(retrieved, start=1):
        snippet = re.sub(r'\s+', ' ', hit.get('text', ''))[:300]
        evidence_lines.append(
            f"[{i}] chunk_id={hit.get('chunk_id')} | category={hit.get('category')} | source={hit.get('source_type')} | score={hit.get('score'):.4f}\n{snippet}"
        )

    evidence_block = '\n\n'.join(evidence_lines) if evidence_lines else 'No evidence retrieved.'

    return f"""You are a Python code-review assistant in a RAG system.

Task:
- Review ONLY the provided diff context and retrieved evidence.
- Focus ONLY on these categories: {', '.join(ALLOWED_CATEGORIES)}.
- Do NOT comment on functionality correctness, security, architecture, or testing strategy.
- If evidence is weak, state uncertainty explicitly in grounded_comment.
- Use predicted_category_hint only as a weak prior, not as a hard constraint.

Context:
- repo: {instance.get('repo')}
- pr_id: {instance.get('pr_id')}
- file_path: {instance.get('file_path')}
- line_number: {instance.get('line_number')}
- predicted_category_hint: {predicted_category}

Diff chunk:
{instance.get('query_text', '')[:2500]}

Retrieved evidence (top-k):
{evidence_block}

Output requirements:
- Return EXACTLY one JSON object only.
- Do not include markdown fences, prose, or extra keys.
- category must be one of: {ALLOWED_CATEGORIES}
- cited_chunk_ids must be a JSON array of chunk IDs (or empty array)

Expected JSON schema:
{{
  "category": "<one allowed category>",
  "grounded_comment": "<single actionable sentence grounded in evidence>",
  "cited_chunk_ids": ["chunk_0001", "chunk_0042"]
}}
"""

In [4]:
missing = [p for p in [EVAL_PATH, INDEX_PATH, META_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required files: ' + ', '.join(str(p) for p in missing) + '\n'
        + 'Run notebooks/embedding_faiss_pipeline.ipynb first to generate FAISS artifacts.'
    )

eval_dataset = load_json(EVAL_PATH)
metadata = load_json(META_PATH)
index = faiss.read_index(str(INDEX_PATH))
model = SentenceTransformer(MODEL_NAME)

instances = build_instances(eval_dataset)
print(f'Evaluation entries: {len(eval_dataset)}')
print(f'Review instances: {len(instances)}')
print(f'FAISS vectors: {index.ntotal}')
print(f'Metadata entries: {len(metadata)}')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3283.83it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation entries: 287
Review instances: 366
FAISS vectors: 1000
Metadata entries: 1000


In [32]:
# Pick one instance for prompt preview
sample_idx = 0
sample = instances[sample_idx]

retrieved = retrieve_dense(sample['query_text'], model, index, metadata, top_k=TOP_K)
predicted_category = predict_category(retrieved)
prompt = build_prompt(sample, retrieved, predicted_category)

print('Gold category:', sample['gold_category'])
print('Predicted category hint:', predicted_category)
print('Top retrieved chunks:')
for i, hit in enumerate(retrieved, start=1):
    print(f"{i}. score={hit['score']:.4f} | chunk_id={hit['chunk_id']} | category={hit['category']} | source={hit['source_type']}")

Gold category: unused_import
Predicted category hint: unused_import
Top retrieved chunks:
1. score=0.7086 | chunk_id=chunk_0425 | category=unused_import | source=review_comment
2. score=0.6730 | chunk_id=chunk_0460 | category=unused_import | source=review_comment
3. score=0.6630 | chunk_id=chunk_0430 | category=unused_import | source=review_comment
4. score=0.6595 | chunk_id=chunk_0961 | category=documentation_formatting | source=review_comment
5. score=0.6574 | chunk_id=chunk_0437 | category=unused_import | source=review_comment


In [6]:
# Prompt preview (send this to your chosen LLM provider)
print(prompt[:5000])

You are a Python code-review assistant in a RAG system.

Task:
- Review ONLY the provided diff context and retrieved evidence.
- Focus ONLY on these categories: indentation, naming_convention, unused_import, mutable_default, documentation_formatting.
- Do NOT comment on functionality correctness, security, architecture, or testing strategy.
- If evidence is weak, say uncertainty explicitly.
- Output exactly one concise review comment.

Context:
- repo: pallets/flask
- pr_id: PR_5775
- file_path: tests/test_helpers.py
- line_number: 238
- predicted_category_hint: unused_import

Diff chunk:
tests/test_helpers.py


class TestStreaming:
    def test_streaming_with_context(self, app, client):
        @app.route("/")
        def index():
    def test_stream_with_context_fails_with_async_route(self):
        import gc

        import pytest

        import flask

        app = flask.Flask(__name__)

        @app.route("/stream")
        async def stream():
            @flask.stream_with_conte

In [7]:
# Build a small batch of retrieval+prompting artifacts for report/demo
num_examples = 10
rows = []
for ins in instances[:num_examples]:
    hits = retrieve_dense(ins['query_text'], model, index, metadata, top_k=TOP_K)
    pred_cat = predict_category(hits)
    rows.append({
        'pr_id': ins['pr_id'],
        'repo': ins['repo'],
        'file_path': ins['file_path'],
        'line_number': ins['line_number'],
        'gold_category': ins['gold_category'],
        'predicted_category_hint': pred_cat,
        'retrieved_chunks': [
            {
                'chunk_id': h['chunk_id'],
                'category': h['category'],
                'source_type': h['source_type'],
                'score': round(h['score'], 4),
            }
            for h in hits
        ],
        'prompt': build_prompt(ins, hits, pred_cat),
    })

with OUT_EXAMPLES.open('w', encoding='utf-8') as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print(f'Saved retrieval+prompting examples: {OUT_EXAMPLES}')

Saved retrieval+prompting examples: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\milestone3\retrieval_prompting_examples.json


## Optional: Connect To An LLM API
If you want end-to-end generated comments, send `prompt` to your LLM provider and parse structured output:
- `category`
- `grounded_comment`
- `cited_chunk_ids`

Keep generation constrained to the project scope (5 categories only) to stay aligned with evaluation setup.

In [11]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env in parent directory
env_path = Path('.') / '.env'
if env_path.exists():
    load_dotenv(env_path)
else:
    print(f"Warning: .env file not found at {env_path}")

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please set it in .env file.")

print("✓ GROQ_API_KEY loaded successfully")

✓ GROQ_API_KEY loaded successfully


In [ ]:
# Install groq client (run once if needed)
# !pip install -q groq python-dotenv

In [ ]:
from groq import Groq
import time

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# Model and rate-limit configuration.
GROQ_MODEL = "openai/gpt-oss-20b"
GROQ_RPM_LIMIT = 30
GROQ_MIN_INTERVAL = 60.0 / GROQ_RPM_LIMIT
GROQ_MAX_RETRIES = 3

_last_groq_request_ts = 0.0

def _wait_for_groq_slot():
    global _last_groq_request_ts
    now = time.time()
    elapsed = now - _last_groq_request_ts
    if elapsed < GROQ_MIN_INTERVAL:
        time.sleep(GROQ_MIN_INTERVAL - elapsed)
    _last_groq_request_ts = time.time()

def _extract_retry_delay(exc: Exception, default_wait: float) -> float:
    # Prefer server-provided retry-after when available.
    try:
        response = getattr(exc, "response", None)
        headers = getattr(response, "headers", {}) if response is not None else {}
        retry_after = headers.get("retry-after") if hasattr(headers, "get") else None
        if retry_after:
            return max(float(retry_after), default_wait)
    except Exception:
        pass

    msg = str(exc)
    m = re.search(r"try again in\s+(\d+(?:\.\d+)?)s", msg, flags=re.IGNORECASE)
    if m:
        return max(float(m.group(1)), default_wait)

    return default_wait

def call_groq_api(prompt: str, max_tokens: int = 512, temperature: float = 0.1, max_retries: int = GROQ_MAX_RETRIES) -> str:
    """
    Call Groq API with RPM-aware pacing and retries.

    Args:
        prompt: The system+user prompt for code review
        max_tokens: Maximum tokens in response
        temperature: Sampling temperature (lower = more deterministic)
        max_retries: Number of retries after the first attempt

    Returns:
        Generated response string, or empty string on final failure
    """
    last_error = None

    for attempt in range(1, max_retries + 2):
        try:
            _wait_for_groq_slot()
            print(f"[API] Attempt {attempt}: model={GROQ_MODEL}, rpm_limit={GROQ_RPM_LIMIT}, max_tokens={max_tokens}")
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "Return exactly one JSON object and nothing else."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=max_tokens,
                temperature=temperature
            )

            if not response.choices:
                print("[API] WARNING: No choices returned")
                if attempt <= max_retries:
                    time.sleep(GROQ_MIN_INTERVAL)
                    continue
                return ""

            msg = response.choices[0].message
            content = msg.content if msg and msg.content else ""
            print(f"[API] content length: {len(content)}")

            if content.strip():
                return content

            print("[API] WARNING: Empty content returned")
            if attempt <= max_retries:
                time.sleep(GROQ_MIN_INTERVAL)
                continue
            return ""

        except Exception as e:
            last_error = e
            print(f"[API] ERROR on attempt {attempt}: {type(e).__name__}: {e}")
            if attempt <= max_retries:
                if "429" in str(e):
                    wait_s = _extract_retry_delay(e, default_wait=GROQ_MIN_INTERVAL * (2 ** attempt))
                else:
                    wait_s = min(8.0, GROQ_MIN_INTERVAL * attempt)
                print(f"[API] Backing off for {wait_s:.1f}s before retry")
                time.sleep(wait_s)
                continue
            break

    if last_error is not None:
        print(f"[API] Final failure after retries: {type(last_error).__name__}: {last_error}")
    return ""

def _strip_code_fences(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```') and cleaned.endswith('```'):
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
        cleaned = re.sub(r'\s*```$', '', cleaned)
    return cleaned.strip()

def parse_groq_output(response: str, debug: bool = False) -> dict:
    """
    Parse structured output from Groq response.
    Extract category, grounded_comment, and cited_chunk_ids.

    Returns dict with parse diagnostics.
    """
    result = {
        'category': None,
        'grounded_comment': None,
        'cited_chunk_ids': [],
        'is_valid': False,
        'parse_status': 'unparsed',
    }

    if not response or not response.strip():
        result['parse_status'] = 'empty_response'
        result['category'] = 'documentation_formatting'
        result['grounded_comment'] = 'Unable to parse response; review required.'
        return result

    cleaned = _strip_code_fences(response)

    # First try strict JSON parsing.
    try:
        payload = json.loads(cleaned)

        # Guard against unexpected top-level types from model outputs.
        if isinstance(payload, str):
            payload = json.loads(payload)

        if isinstance(payload, list):
            payload = payload[0] if payload else {}

        if isinstance(payload, dict):
            cat = str(payload.get('category', '')).strip()
            comment = str(payload.get('grounded_comment', '')).strip()
            ids = payload.get('cited_chunk_ids', [])

            if isinstance(ids, str):
                ids = [x.strip() for x in ids.split(',') if x.strip()]
            elif not isinstance(ids, list):
                ids = []

            ids = [str(x).strip() for x in ids if str(x).strip()]

            if cat in ALLOWED_CATEGORIES and comment:
                result['category'] = cat
                result['grounded_comment'] = comment
                result['cited_chunk_ids'] = ids
                result['is_valid'] = True
                result['parse_status'] = 'json_ok'
                return result

        result['parse_status'] = 'json_missing_fields'
    except Exception:
        result['parse_status'] = 'json_parse_failed'

    # Fallback parser for line-based output.
    lines = cleaned.split('\n')
    if debug:
        print(f"[DEBUG] Total lines in response: {len(lines)}")
        for idx, line in enumerate(lines):
            print(f"[DEBUG] Line {idx}: {repr(line)}")

    for line in lines:
        line = line.strip()
        if line.startswith('category:') or line.startswith('- category:'):
            cat = line.replace('- category:', '').replace('category:', '').strip().strip('`*/')
            if cat in ALLOWED_CATEGORIES:
                result['category'] = cat
        elif line.startswith('grounded_comment:') or line.startswith('- grounded_comment:'):
            comment = line.replace('- grounded_comment:', '').replace('grounded_comment:', '').strip().strip('`*/')
            result['grounded_comment'] = comment
        elif line.startswith('cited_chunk_ids:') or line.startswith('- cited_chunk_ids:'):
            ids_str = line.replace('- cited_chunk_ids:', '').replace('cited_chunk_ids:', '').strip().strip('`*/')
            if ids_str and ids_str.lower() != 'none':
                result['cited_chunk_ids'] = [x.strip() for x in ids_str.split(',') if x.strip()]

    if result['category'] in ALLOWED_CATEGORIES and result['grounded_comment']:
        result['is_valid'] = True
        result['parse_status'] = 'line_parse_ok'
        return result

    result['category'] = result['category'] if result['category'] in ALLOWED_CATEGORIES else 'documentation_formatting'
    if not result['grounded_comment']:
        result['grounded_comment'] = 'Unable to parse response; review required.'
    if result['parse_status'] in {'json_parse_failed', 'json_missing_fields'}:
        result['parse_status'] = f"{result['parse_status']}_and_line_parse_failed"
    else:
        result['parse_status'] = 'line_parse_failed'
    return result

print("✓ Groq client initialized with 30 RPM-safe pacing and parser helpers")

✓ Groq client initialized and helper functions defined


In [34]:
# Test Groq API on a single example
test_response = call_groq_api(prompt)
print("Groq raw response:")
print(test_response)
print("\n" + "="*80 + "\n")

parsed = parse_groq_output(test_response, debug=True)
print("Parsed output:")
print(f"Category: {parsed['category']}")
print(f"Comment: {parsed['grounded_comment']}")
print(f"Cited chunks: {parsed['cited_chunk_ids']}")
print(f"Valid parse: {parsed['is_valid']}")
print(f"Parse status: {parsed['parse_status']}")

[API] Attempt 1: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WARNING: Empty content returned
[API] Attempt 2: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WARNING: Empty content returned
[API] Attempt 3: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WARNING: Empty content returned
Groq raw response:



Parsed output:
Category: documentation_formatting
Comment: Unable to parse response; review required.
Cited chunks: []
Valid parse: False
Parse status: empty_response


In [ ]:
from collections import Counter

# Full batch inference with Groq API
num_batch = 12  # Keep conservative by default to stay within RPM/testing budget
llm_results = []
parse_status_counter = Counter()

print(f"Running batch inference on {min(num_batch, len(instances))} instances...")
print(f"Model: {GROQ_MODEL}")
print(f"Rate limit policy: {GROQ_RPM_LIMIT} RPM (~{GROQ_MIN_INTERVAL:.2f}s min interval between calls)\n")

for idx, ins in enumerate(instances[:num_batch], start=1):
    print(f"[{idx}/{num_batch}] Processing {ins['pr_id']} from {ins['repo']}...", end=" ")

    # Retrieve evidence
    hits = retrieve_dense(ins['query_text'], model, index, metadata, top_k=TOP_K)
    pred_cat = predict_category(hits)

    # Build prompt
    prompt_text = build_prompt(ins, hits, pred_cat)

    # Call Groq API
    groq_response = call_groq_api(prompt_text)

    # Parse output
    parsed_output = parse_groq_output(groq_response)
    parse_status_counter[parsed_output['parse_status']] += 1

    # Use strict prediction only when parse is valid
    strict_pred = parsed_output['category'] if parsed_output['is_valid'] else None

    # Build result object
    res = {
        'pr_id': ins['pr_id'],
        'repo': ins['repo'],
        'file_path': ins['file_path'],
        'line_number': ins['line_number'],
        'gold_category': ins['gold_category'],
        'gold_comment': ins['gold_comment'],
        'predicted_category_hint': pred_cat,
        'groq_predicted_category': parsed_output['category'],
        'groq_predicted_category_valid_only': strict_pred,
        'groq_grounded_comment': parsed_output['grounded_comment'],
        'groq_cited_chunks': parsed_output['cited_chunk_ids'],
        'llm_parse_valid': parsed_output['is_valid'],
        'llm_parse_status': parsed_output['parse_status'],
        'llm_raw_response': groq_response,
        'retrieval_backend': 'faiss_dense',
        'generation_backend': 'groq_llm',
        'retrieved_chunks': [
            {
                'chunk_id': h['chunk_id'],
                'category': h['category'],
                'source_type': h['source_type'],
                'score': round(h['score'], 4),
                'text': h['text'][:200]  # Include snippet for reference
            }
            for h in hits
        ],
    }

    llm_results.append(res)
    print(f"✓ ({parsed_output['parse_status']})")

print(f"\nCompleted {len(llm_results)} inferences")
print("Parse status summary:")
for k, v in parse_status_counter.items():
    print(f"  {k}: {v}")

Running batch inference on 20 instances...
Model: openai/gpt-oss-20b

[1/20] Processing PR_5775 from pallets/flask... [API] Attempt 1: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 186
✓ (json_ok)
[2/20] Processing PR_4992 from pallets/flask... [API] Attempt 1: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WARNING: Empty content returned
[API] Attempt 2: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WARNING: Empty content returned
[API] Attempt 3: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 139
✓ (json_ok)
[3/20] Processing PR_4682 from pallets/flask... [API] Attempt 1: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 201
✓ (json_ok)
[4/20] Processing PR_4560 from pallets/flask... [API] Attempt 1: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WARNING: Empty content returned
[API] Attempt 2: model=openai/gpt-oss-20b, max_tokens=512
[API] content length: 0
[API] WA

KeyboardInterrupt: 

In [ ]:
# Compute metrics (Groq predictions vs gold)
gold_labels = [r['gold_category'] for r in llm_results]
groq_preds_raw = [r['groq_predicted_category'] for r in llm_results]

valid_results = [
    r for r in llm_results
    if r.get('llm_parse_valid') and r.get('groq_predicted_category_valid_only') in ALLOWED_CATEGORIES
]
gold_labels_valid = [r['gold_category'] for r in valid_results]
groq_preds_valid = [r['groq_predicted_category_valid_only'] for r in valid_results]

def compute_metrics(gold: list, pred: list) -> dict:
    """Compute accuracy, precision, recall, F1 per category"""
    tp = Counter()
    fp = Counter()
    fn = Counter()

    for g, p in zip(gold, pred):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1

    metrics = {
        'accuracy': sum(1 for g, p in zip(gold, pred) if g == p) / len(gold) if gold else 0,
        'per_class': {}
    }

    for c in ALLOWED_CATEGORIES:
        p_val = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0
        r_val = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0
        f1_val = 2 * p_val * r_val / (p_val + r_val) if (p_val + r_val) > 0 else 0
        metrics['per_class'][c] = {
            'precision': round(p_val, 4),
            'recall': round(r_val, 4),
            'f1': round(f1_val, 4),
            'support': sum(1 for x in gold if x == c)
        }

    return metrics

def summarize_parse_quality(results: list[dict]) -> dict:
    status_counts = Counter(r.get('llm_parse_status', 'unknown') for r in results)
    valid_count = sum(1 for r in results if r.get('llm_parse_valid'))
    total = len(results)
    return {
        'total': total,
        'valid_count': valid_count,
        'invalid_count': total - valid_count,
        'valid_rate': round(valid_count / total, 4) if total else 0,
        'status_counts': dict(status_counts),
    }

metrics_raw = compute_metrics(gold_labels, groq_preds_raw)
metrics_valid_only = compute_metrics(gold_labels_valid, groq_preds_valid)
parse_quality = summarize_parse_quality(llm_results)

print("="*80)
print("GROQ LLM Results on Batch")
print("="*80)
print(f"Total samples: {parse_quality['total']}")
print(f"Valid parsed responses: {parse_quality['valid_count']}")
print(f"Invalid/empty responses: {parse_quality['invalid_count']}")
print(f"Valid response rate: {parse_quality['valid_rate']:.4f}")
print("\nParse status counts:")
for k, v in parse_quality['status_counts'].items():
    print(f"  {k}: {v}")

print("\n" + "="*80)
print("RAW METRICS (includes fallback predictions)")
print("="*80)
print(f"Accuracy: {metrics_raw['accuracy']:.4f}")
for cat, scores in metrics_raw['per_class'].items():
    print(f"  {cat}: P={scores['precision']:.4f} R={scores['recall']:.4f} F1={scores['f1']:.4f} support={scores['support']}")

print("\n" + "="*80)
print("VALID-ONLY METRICS (only parse-valid responses)")
print("="*80)
if valid_results:
    print(f"Evaluated valid responses: {len(valid_results)}")
    print(f"Accuracy: {metrics_valid_only['accuracy']:.4f}")
    for cat, scores in metrics_valid_only['per_class'].items():
        print(f"  {cat}: P={scores['precision']:.4f} R={scores['recall']:.4f} F1={scores['f1']:.4f} support={scores['support']}")
else:
    print("No valid parsed responses; valid-only metrics are unavailable.")

GROQ LLM Results on Batch:

Accuracy: 0.3000

Per-category metrics:
  indentation:
    Precision: 0.0000
    Recall: 0.0000
    F1: 0.0000
    Support: 3
  naming_convention:
    Precision: 0.0000
    Recall: 0.0000
    F1: 0.0000
    Support: 5
  unused_import:
    Precision: 0.0000
    Recall: 0.0000
    F1: 0.0000
    Support: 2
  mutable_default:
    Precision: 0.0000
    Recall: 0.0000
    F1: 0.0000
    Support: 0
  documentation_formatting:
    Precision: 0.5455
    Recall: 0.6000
    F1: 0.5714
    Support: 10


In [29]:
# Sample output visualization
print("\n" + "="*80)
print("Sample Outputs (First 3 Examples):")
print("="*80 + "\n")

for i, result in enumerate(llm_results[:3], start=1):
    print(f"Example {i}:")
    print(f"  PR ID: {result['pr_id']}")
    print(f"  Repo: {result['repo']}")
    print(f"  File: {result['file_path']}")
    print(f"  Line: {result['line_number']}")
    print(f"\n  Gold category: {result['gold_category']}")
    print(f"  Gold comment: {result['gold_comment'][:150]}...")
    print(f"\n  Groq predicted: {result['groq_predicted_category']}")
    print(f"  Groq comment: {result['groq_grounded_comment']}")
    print(f"  Cited chunks: {result['groq_cited_chunks']}")
    print(f"  Match: {'✓' if result['gold_category'] == result['groq_predicted_category'] else '✗'}")
    print("\n" + "-"*80 + "\n")


Sample Outputs (First 3 Examples):

Example 1:
  PR ID: PR_5775
  Repo: pallets/flask
  File: tests/test_helpers.py
  Line: 238

  Gold category: unused_import
  Gold comment: These imports should not be local to this function, and should not have spaces in between each one....

  Groq predicted: documentation_formatting
  Groq comment: Unable to parse response; review required.
  Cited chunks: []
  Match: ✗

--------------------------------------------------------------------------------

Example 2:
  PR ID: PR_4992
  Repo: pallets/flask
  File: src/flask/config.py
  Line: 259

  Gold category: documentation_formatting
  Gold comment: Need to add documentation for the parameter....

  Groq predicted: documentation_formatting
  Groq comment: Unable to parse response; review required.
  Cited chunks: []
  Match: ✓

--------------------------------------------------------------------------------

Example 3:
  PR ID: PR_4682
  Repo: pallets/flask
  File: src/flask/globals.py
  Line: 73



In [ ]:
# Save full results with Groq predictions to output file
OUT_LLM_RESULTS = OUT_DIR / 'llm_results.json'

output_payload = {
    'model': GROQ_MODEL,
    'num_samples': len(llm_results),
    'run_config': {
        'retrieval_backend': 'faiss_dense',
        'generation_backend': 'groq_llm',
        'groq_rpm_limit': GROQ_RPM_LIMIT,
        'groq_min_interval_seconds': round(GROQ_MIN_INTERVAL, 4),
        'groq_max_retries': GROQ_MAX_RETRIES,
    },
    'parse_quality': parse_quality,
    'metrics_raw': {
        'accuracy': round(metrics_raw['accuracy'], 4),
        'per_class': metrics_raw['per_class']
    },
    'metrics_valid_only': {
        'accuracy': round(metrics_valid_only['accuracy'], 4),
        'per_class': metrics_valid_only['per_class'],
        'num_valid_samples': len(valid_results)
    },
    'results': llm_results
}

with OUT_LLM_RESULTS.open('w', encoding='utf-8') as f:
    json.dump(output_payload, f, indent=2, ensure_ascii=False)

print(f"\n✓ Full results saved to: {OUT_LLM_RESULTS}")
print(f"  Total samples: {len(llm_results)}")
print(f"  Valid parsed responses: {parse_quality['valid_count']}")
print(f"  Raw accuracy: {metrics_raw['accuracy']:.4f}")
print(f"  Valid-only accuracy: {metrics_valid_only['accuracy']:.4f}")
print(f"  RPM policy used: {GROQ_RPM_LIMIT} RPM")


✓ Full results saved to: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\milestone3\groq_llm_results.json
  Total samples: 20
  Accuracy: 0.3000
